# Pig Posture Recognition - V5 Inference

**Ensemble: DINOv2 Large (ViT, @518px) + ConvNeXt V2 Large (CNN, @384px)**

**Pipeline:**
1. Pro Checkpoint: Modell laden (Pos-Embed-Resampling falls noetig fuer ViT)
2. Test-Time BN Adaptation (nur ConvNeXt)
3. 6-fach TTA mit Flip-Korrektur fuer Lateral_left/right
4. Per-Architektur gewichtetes Ensemble (ViT=1.0, CNN=0.5)
5. Submission CSV + Pseudo-Label Export

## Configuration

In [1]:
import os

TAG = "T2"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

_candidates = [
    "multiview_pig_posture_recognition",
    "./multiview_pig_posture_recognition",
    "/datasets/multi-view-pig-posture-recognition",
    "/multi-view-pig-posture-recognition",
]
DATA_ROOT = None
for _p in _candidates:
    if os.path.isdir(_p):
        DATA_ROOT = _p
        break
assert DATA_ROOT is not None
print(f"DATA_ROOT = {os.path.abspath(DATA_ROOT)}")

TEST_CSV  = os.path.join(DATA_ROOT, "test.csv")
IMG_DIR   = os.path.join(DATA_ROOT, "test_images")

CKPT_DIR  = f"runs/v5_{TAG.lower()}"
CKPT_PATHS = sorted([
    os.path.join(CKPT_DIR, f) for f in os.listdir(CKPT_DIR)
    if f.startswith("best_") and f.endswith(".pth")
])

OUTPUT_FILE = f"{TAG}_v5_submission.csv"

# --- Inferenz-Aufloesung pro Architektur-Prefix ---
# Matched Training-Aufloesung -> kein pos_embed-Mismatch noetig.
# (Du kannst hoeher gehen, profitiert ggf. um 0.5%, kostet aber VRAM)
INFER_IMG_SIZE = {
    "dinov2l":  518,
    "convnextv2l":  384,
}

# Inferenz auf 4x V100 32GB
BATCH_SIZE   = 32
NUM_WORKERS  = 16
USE_TTA      = True
PAD_RATIO    = 0.1
NUM_CLASSES  = 5

CLASS_NAMES = ["Lateral_lying_left", "Lateral_lying_right",
               "Sitting", "Standing", "Sternal_lying"]

# --- Test-Time BN Adaptation (nur fuer ConvNeXt relevant) ---
ADAPT_BN         = True
BN_ADAPT_BATCHES = 50

# --- Pseudo-Label Export ---
EXPORT_PSEUDO_LABELS   = True
PSEUDO_MODE            = "top_k"
PSEUDO_THRESHOLD       = 0.85
PSEUDO_TOP_K_PER_CLASS = 200
PSEUDO_MIN_CONFIDENCE  = 0.60
PSEUDO_OUTPUT          = f"pseudo_labels_{TAG.lower()}_v5.csv"

# --- Ensemble-Gewichtung pro Architektur ---
# ConvNeXt V2 Large ist deutlich staerker als V1 Base -> hoeheres Gewicht als V4
ENSEMBLE_WEIGHTS = {"dinov2l": 1.0, "convnextv2l": 0.5}

print(f"Tag: {TAG}  |  Version: v5  |  Checkpoints: {len(CKPT_PATHS)}")
for p in CKPT_PATHS:
    print(f"  - {os.path.basename(p)}")

DATA_ROOT = /datasets/multi-view-pig-posture-recognition
Tag: T2  |  Version: v5  |  Checkpoints: 6
  - best_convnextv2l_fold_1.pth
  - best_convnextv2l_fold_2.pth
  - best_convnextv2l_fold_3.pth
  - best_dinov2l_fold_1.pth
  - best_dinov2l_fold_2.pth
  - best_dinov2l_fold_3.pth


## Imports

In [2]:
import os, ast, re
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import torchvision.transforms as T
import timm

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

<jemalloc>: Unsupported system page size


Device: cuda


## Test Dataset

In [3]:
class PigTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.1):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform:
            crop = self.transform(crop)
        return crop, row["row_id"]

## Test-Time BN Adaptation

Nur relevant fuer ConvNeXt V2 (BatchNorm). DINOv2/EVA-02/ViT wird automatisch uebersprungen (LayerNorm).

In [4]:
def adapt_batch_norm(model, loader, device, n_batches=50):
    bn_layers = [m for m in model.modules()
                 if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.SyncBatchNorm))]
    if not bn_layers:
        print("    Keine BatchNorm Layer - uebersprungen.")
        return
    for bn in bn_layers:
        bn.running_mean.zero_()
        bn.running_var.fill_(1)
        bn.momentum = None
    model.train()
    with torch.no_grad():
        for i, (imgs, _) in enumerate(tqdm(loader, desc="    BN-Adapt", leave=False)):
            if i >= n_batches: break
            model(imgs.to(device))
    model.eval()
    print(f"    BN auf {min(n_batches, len(loader))} Batches adaptiert ({len(bn_layers)} Layer)")

## TTA mit Flip-Korrektur

6 Views: Original, HFlip, +32px Crop, +32px HFlip, +64px Crop, 0.75 Downscale.
Flip vertauscht Lateral_lying_left (0) <-> Lateral_lying_right (1) in den Probabilities.

In [5]:
NORM = [[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]]

def build_tta_configs(img_size):
    S = img_size
    return [
        (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.RandomHorizontalFlip(p=1.0),
                    T.ToTensor(), T.Normalize(*NORM)]), True),
        (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0),
                    T.ToTensor(), T.Normalize(*NORM)]), True),
        (T.Compose([T.Resize((S+64, S+64), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((int(S*0.75), int(S*0.75))),
                    T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
    ]

## Multi-Arch Inference

In [6]:
test_df = pd.read_csv(TEST_CSV)
print(f"Test-Instanzen: {len(test_df)}")


@torch.no_grad()
def predict_tta(model, df, img_dir, tta_configs, batch_size):
    all_probs = []
    for i, (tf, is_flipped) in enumerate(tta_configs):
        ds = PigTestDataset(df, img_dir, transform=tf, pad_ratio=PAD_RATIO)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        probs = []
        for imgs, _ in tqdm(loader, desc=f"  TTA {i+1}/{len(tta_configs)}", leave=False):
            with autocast():
                logits = model(imgs.to(DEVICE))
            p = F.softmax(logits, dim=1).cpu().numpy()
            if is_flipped:
                p[:, [0, 1]] = p[:, [1, 0]]
            probs.append(p)
        all_probs.append(np.vstack(probs))
    return np.mean(all_probs, axis=0)


def infer_prefix(ckpt_path):
    """Extrahiere Arch-Prefix aus Dateiname (z.B. 'best_dinov2l_fold_1.pth' -> 'dinov2l')."""
    name = os.path.basename(ckpt_path)
    m = re.match(r"best_([a-zA-Z0-9]+)_fold_\d+\.pth", name)
    return m.group(1) if m else "unknown"


def load_vit_with_resampling(name, ckpt, num_classes, infer_size):
    """Lade ViT-Checkpoint und interpoliere pos_embed falls Aufloesungen differieren."""
    model = timm.create_model(name, pretrained=False, num_classes=num_classes, img_size=infer_size)
    state_dict = dict(ckpt["model"])

    if "pos_embed" in state_dict:
        old_pe = state_dict["pos_embed"]
        new_pe = model.pos_embed
        if old_pe.shape != new_pe.shape:
            try:
                from timm.layers import resample_abs_pos_embed
            except ImportError:
                from timm.models.layers import resample_abs_pos_embed
            new_grid = model.patch_embed.grid_size
            num_prefix = getattr(model, "num_prefix_tokens", 1)
            print(f"    Interpoliere pos_embed: {tuple(old_pe.shape)} -> {tuple(new_pe.shape)} "
                  f"(grid={new_grid}, prefix_tokens={num_prefix})")
            state_dict["pos_embed"] = resample_abs_pos_embed(
                old_pe, new_size=new_grid, num_prefix_tokens=num_prefix
            )

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"    Missing keys: {missing[:3]}{'...' if len(missing)>3 else ''}")
    if unexpected:
        print(f"    Unexpected keys: {unexpected[:3]}{'...' if len(unexpected)>3 else ''}")
    return model


ensemble_probs_weighted = []
ensemble_weights_used = []

for idx, path in enumerate(CKPT_PATHS):
    prefix = infer_prefix(path)
    print(f"\nModell {idx+1}/{len(CKPT_PATHS)}: {os.path.basename(path)} (arch={prefix})")

    ckpt = torch.load(path, map_location="cpu")
    name = ckpt.get("model_name", "unknown")
    ckpt_img_size = ckpt.get("img_size", 384)
    infer_size = INFER_IMG_SIZE.get(prefix, ckpt_img_size)

    print(f"  {name}  |  Val F1: {ckpt.get('val_f1', 0):.4f}  |  "
          f"Train@{ckpt_img_size}px, Infer@{infer_size}px")

    # ViT-Familien (DINOv2/EVA-02): mit img_size + pos_embed resampling
    # ConvNeXt V2: kein img_size Argument, keine pos_embed
    try:
        model = load_vit_with_resampling(name, ckpt, NUM_CLASSES, infer_size)
    except TypeError:
        model = timm.create_model(name, pretrained=False, num_classes=NUM_CLASSES)
        model.load_state_dict(ckpt["model"])

    model.to(DEVICE).eval()

    # BN Adaptation (nur BatchNorm-Modelle)
    if ADAPT_BN:
        bn_tf = T.Compose([
            T.Resize((infer_size, infer_size), interpolation=T.InterpolationMode.BICUBIC),
            T.ToTensor(), T.Normalize(*NORM),
        ])
        bn_ds = PigTestDataset(test_df, IMG_DIR, transform=bn_tf, pad_ratio=PAD_RATIO)
        bn_loader = DataLoader(bn_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
        adapt_batch_norm(model, bn_loader, DEVICE, n_batches=BN_ADAPT_BATCHES)

    tta_cfg = build_tta_configs(infer_size) if USE_TTA else [build_tta_configs(infer_size)[0]]
    fold_probs = predict_tta(model, test_df, IMG_DIR, tta_cfg, BATCH_SIZE)

    w = ENSEMBLE_WEIGHTS.get(prefix, 1.0) if ENSEMBLE_WEIGHTS else 1.0
    ensemble_probs_weighted.append(fold_probs * w)
    ensemble_weights_used.append(w)

    del model
    torch.cuda.empty_cache()

final_probs = np.sum(ensemble_probs_weighted, axis=0) / max(sum(ensemble_weights_used), 1e-6)
predictions = final_probs.argmax(axis=1)

print(f"\n{len(predictions)} Vorhersagen aus {len(CKPT_PATHS)} Modellen")

Test-Instanzen: 11708

Modell 1/6: best_convnextv2l_fold_1.pth (arch=convnextv2l)
  convnextv2_large.fcmae_ft_in22k_in1k_384  |  Val F1: 0.7729  |  Train@384px, Infer@384px
    Keine BatchNorm Layer - uebersprungen.


  TTA 1/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 2/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 3/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 4/6:   0%|          | 0/366 [00:00<?, ?it/s]

...

  TTA 3/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 4/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 5/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 6/6:   0%|          | 0/366 [00:00<?, ?it/s]


11708 Vorhersagen aus 6 Modellen


## Submission

In [7]:
submission = pd.DataFrame({
    "row_id": test_df["row_id"].values,
    "class_id": predictions.astype(int)
})
submission.to_csv(OUTPUT_FILE, index=False)

assert list(submission.columns) == ["row_id", "class_id"]
assert set(submission["class_id"].unique()).issubset(set(range(5)))
assert len(submission) == len(test_df)

print(f"Submission: {OUTPUT_FILE} ({len(submission)} Zeilen)\n")
print(f"Verteilung:")
for c in range(NUM_CLASSES):
    cnt = (submission["class_id"] == c).sum()
    pct = 100 * cnt / len(submission)
    bar = "#" * int(30 * cnt / len(submission))
    print(f"  {c} - {CLASS_NAMES[c]:<22} {bar:<30} {cnt:>5} ({pct:.1f}%)")

submission.head(10)

Submission: T2_v5_submission.csv (11708 Zeilen)

Verteilung:
  0 - Lateral_lying_left     ###                             1337 (11.4%)
  1 - Lateral_lying_right    ###                             1528 (13.1%)
  2 - Sitting                #                                428 (3.7%)
  3 - Standing               ###############                 5930 (50.6%)
  4 - Sternal_lying          ######                          2485 (21.2%)


,row_id,class_id
0,test_pen1_tur_cam1_20250920_174649_0000,3
1,test_pen1_tur_cam1_20250920_174649_0001,1
2,test_pen1_tur_cam1_20250920_174649_0002,1
3,test_pen1_tur_cam1_20250920_174649_0003,1
4,test_pen1_tur_cam1_20250920_174649_0004,0
5,test_pen1_tur_cam1_20250920_174649_0005,1
6,test_pen1_tur_cam1_20250920_174649_0006,1
7,test_pen1_tur_cam1_20250920_174649_0007,1
8,test_pen1_tur_cam1_20250920_174649_0008,0
9,test_pen1_tur_cam1_20250921_050022_0000,0


## Class-balanced Pseudo-Label Export

Top-K pro Klasse, mit min Confidence, fuer balancierten Trainings-Boost im naechsten Run.

In [8]:
if EXPORT_PSEUDO_LABELS:
    max_probs = final_probs.max(axis=1)
    argmax = final_probs.argmax(axis=1)

    if PSEUDO_MODE == "top_k":
        selected_idx = []
        print(f"Class-balanced Top-{PSEUDO_TOP_K_PER_CLASS} pro Klasse "
              f"(min conf {PSEUDO_MIN_CONFIDENCE}):")
        for c in range(NUM_CLASSES):
            cand_mask = (argmax == c) & (max_probs >= PSEUDO_MIN_CONFIDENCE)
            cand_idx = np.where(cand_mask)[0]
            cand_idx = cand_idx[np.argsort(-max_probs[cand_idx])]
            chosen = cand_idx[:PSEUDO_TOP_K_PER_CLASS]
            selected_idx.extend(chosen.tolist())
            if len(chosen) > 0:
                print(f"  {c} - {CLASS_NAMES[c]:<22} "
                      f"{len(chosen):>4} ausgewaehlt "
                      f"(min conf={max_probs[chosen].min():.3f}, "
                      f"max conf={max_probs[chosen].max():.3f})")
            else:
                print(f"  {c} - {CLASS_NAMES[c]:<22} KEINE Kandidaten")
        selected_idx = np.array(sorted(selected_idx), dtype=int)
        mask = np.zeros(len(test_df), dtype=bool)
        mask[selected_idx] = True
    else:
        mask = max_probs >= PSEUDO_THRESHOLD

    pseudo_df = test_df[mask].copy()
    pseudo_df["class_id"] = predictions[mask].astype(int)
    pseudo_df["confidence"] = max_probs[mask]
    pseudo_df.to_csv(PSEUDO_OUTPUT, index=False)

    n = mask.sum()
    print(f"\nPseudo-Labels: {PSEUDO_OUTPUT}")
    print(f"  {n} von {len(test_df)} ({100*n/len(test_df):.1f}%)")
    print(f"  Verteilung:")
    for c in range(NUM_CLASSES):
        cnt = (pseudo_df["class_id"] == c).sum()
        print(f"    {c} - {CLASS_NAMES[c]:<22} {cnt:>4}")
else:
    print("Pseudo-Label Export deaktiviert.")

Class-balanced Top-200 pro Klasse (min conf 0.6):
  0 - Lateral_lying_left      200 ausgewaehlt (min conf=0.931, max conf=0.944)
  1 - Lateral_lying_right     200 ausgewaehlt (min conf=0.929, max conf=0.941)
  2 - Sitting                 200 ausgewaehlt (min conf=0.862, max conf=0.995)
  3 - Standing                200 ausgewaehlt (min conf=0.818, max conf=0.828)
  4 - Sternal_lying           200 ausgewaehlt (min conf=0.838, max conf=0.868)

Pseudo-Labels: pseudo_labels_t2_v5.csv
  1000 von 11708 (8.5%)
  Verteilung:
    0 - Lateral_lying_left      200
    1 - Lateral_lying_right     200
    2 - Sitting                 200
    3 - Standing                200
    4 - Sternal_lying           200
